In [1]:
import os
import numpy as np
import torch

print(torch.__version__)

1.10.0


First we select the folder with the featuremaps we want to generate data from

In [2]:
root = '/home/mingdayang/mmdetection3d/outputs/inference/baseline/unibev_val_LC_full_mini_nuscenes'

if os.path.exists(root):
    print(f"The folder '{root}' exists.")
else:
    print(f"The folder '{root}' does not exist.")

# Collect only folders that contain vis_data.pt
folders = sorted(
    f for f in os.listdir(root)
    if os.path.exists(os.path.join(root, f, 'vis_data.pt'))
)
print(folders)
print(len(folders))


The folder '/home/mingdayang/mmdetection3d/outputs/inference/baseline/unibev_val_LC_full_mini_nuscenes' exists.
['n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151603547590', 'n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151604048025', 'n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151604547893', 'n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151605047769', 'n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151605548192', 'n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151606048630', 'n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151606549066', 'n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151607048933', 'n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151607548824', 'n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151608048151', 'n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151608548020', 'n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151609047890', 'n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151609547766', 'n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151609947025', 'n008-2018-08-01-15

Then we loop through each folder, grab the vis_data.pt and put them into a tensor. Since torch tensor needs a predefined size, we can generate that using the length of `file_list`.

In [3]:
# Peek first file to get shapes/dtypes
first = torch.load(os.path.join(root, folders[0], 'vis_data.pt'))
img0 = first['img_bev_embed']
pts0 = first['pts_bev_embed']
print(first.keys())
print(first['lidar_file_name'])
img_shape, img_dtype = img0.shape, img0.dtype
pts_shape, pts_dtype = pts0.shape, pts0.dtype

print("img shape:", img_shape, "dtype:", img_dtype)
print("pts shape:", pts_shape, "dtype:", pts_dtype)

# Preallocate
img_all = torch.empty((len(folders),) + img_shape, dtype=img_dtype)
pts_all = torch.empty((len(folders),) + pts_shape, dtype=pts_dtype)
print("Amount of folders fount:", len(folders))
print("Preallocated img_all shape:", img_all.shape)
print("Preallocated pts_all shape:", pts_all.shape)

lidar_file_names = []

dict_keys(['ori_img_bev_embed', 'ori_pts_bev_embed', 'feature_weights', 'channel_weights_norm', 'img_norm_weights', 'pts_norm_weights', 'fused_bev_embed', 'img_mlvl_feats', 'img_bev_embed', 'pts_mlvl_feats', 'pts_bev_embed', 'query', 'bev_queries', 'bev_pos', 'query_pos', 'reference_points', 'lidar_file_name'])
n008-2018-08-01-15-16-36-0400__LIDAR_TOP__1533151603547590
img shape: torch.Size([1, 40000, 256]) dtype: torch.float32
pts shape: torch.Size([1, 40000, 256]) dtype: torch.float32
Amount of folders fount: 81
Preallocated img_all shape: torch.Size([81, 1, 40000, 256])
Preallocated pts_all shape: torch.Size([81, 1, 40000, 256])


In [4]:
for i, file in enumerate(folders):
    full_path = os.path.join(root, file, 'vis_data.pt')
    dictionary = torch.load(full_path)
    img_all[i] = dictionary['img_bev_embed']
    pts_all[i] = dictionary['pts_bev_embed']
    if (i+1) % 10 == 0:
        print(f"Loaded {i+1}/{len(folders)}")

print("Done loading all data.")

Loaded 10/81
Loaded 20/81
Loaded 30/81
Loaded 40/81
Loaded 50/81
Loaded 60/81
Loaded 70/81
Loaded 80/81
Done loading all data.


In [6]:
#Squeeze dimension
img_all = img_all.squeeze(1)
pts_all = pts_all.squeeze(1)
print("Squeezed img_all shape:", img_all.shape)
print("Squeezed pts_all shape:", pts_all.shape)

Squeezed img_all shape: torch.Size([81, 40000, 256])
Squeezed pts_all shape: torch.Size([81, 40000, 256])


In [7]:
# inspect found tensors
print("img_all shape:", img_all.shape, "dtype:", img_all.dtype)
print("pts_all shape:", pts_all.shape, "dtype:", pts_all.dtype)

print("Select view of img_all[0]:", img_all[0])

img_all shape: torch.Size([81, 40000, 256]) dtype: torch.float32
pts_all shape: torch.Size([81, 40000, 256]) dtype: torch.float32
Select view of img_all[0]: tensor([[-0.1015, -0.1078,  0.0743,  ...,  0.2083,  0.0666, -0.0093],
        [-0.0897, -0.1131,  0.0603,  ...,  0.2156,  0.0669, -0.0127],
        [-0.0961, -0.1898,  0.0687,  ...,  0.2162,  0.0720, -0.0277],
        ...,
        [ 0.2983,  0.2987,  0.0489,  ...,  0.3283,  0.0788,  0.0644],
        [ 0.3013,  0.2891,  0.0425,  ...,  0.3316,  0.0848,  0.0586],
        [ 0.3072,  0.2842,  0.0503,  ...,  0.3252,  0.0829,  0.0638]])


In [9]:
# Save the combined tensors
data_folder = '/home/mingdayang/mmdetection3d/mapping_test/'
torch.save({'img_bev_embed': img_all, 'lidar_file_name': folders}, os.path.join(data_folder, 'all_img_bev.pt'))
torch.save({'pts_bev_embed': pts_all, 'lidar_file_name': folders}, os.path.join(data_folder, 'all_pts_bev.pt'))
print("Saved combined tensors to disk.")

Saved combined tensors to disk.


In [14]:
d = torch.load(os.path.join(data_folder, 'all_img_bev.pt'))
print(d['img_bev_embed'])

tensor([[[[-1.0155e-01, -1.0783e-01,  7.4267e-02,  ...,  2.0826e-01,
            6.6576e-02, -9.3027e-03],
          [-8.9726e-02, -1.1311e-01,  6.0293e-02,  ...,  2.1556e-01,
            6.6872e-02, -1.2682e-02],
          [-9.6111e-02, -1.8983e-01,  6.8685e-02,  ...,  2.1616e-01,
            7.2010e-02, -2.7745e-02],
          ...,
          [ 2.9830e-01,  2.9875e-01,  4.8851e-02,  ...,  3.2833e-01,
            7.8780e-02,  6.4390e-02],
          [ 3.0128e-01,  2.8913e-01,  4.2466e-02,  ...,  3.3161e-01,
            8.4799e-02,  5.8628e-02],
          [ 3.0716e-01,  2.8416e-01,  5.0311e-02,  ...,  3.2524e-01,
            8.2926e-02,  6.3762e-02]]],


        [[[-6.7744e-02, -4.1755e-01,  1.6819e-01,  ...,  3.6113e-01,
            5.2394e-02, -1.0108e-01],
          [-3.4708e-02, -4.0342e-01,  1.6600e-01,  ...,  3.6967e-01,
            5.1502e-02, -1.0065e-01],
          [-6.8644e-03, -3.9745e-01,  1.6894e-01,  ...,  3.6446e-01,
            5.0225e-02, -1.0809e-01],
          ...,
   

In [44]:
print(d['img_bev_embed'].shape)

torch.Size([81, 1, 40000, 256])


Here is code to generate training and testing .pt files seperately

In [7]:
# Create train/test split
train_split = int(0.8 * len(img_all)) # 80% of data used for training set, 20% for testing 
X_train, y_train = img_all[:train_split], pts_all[:train_split]
X_test, y_test = img_all[train_split:], pts_all[train_split:]

print(len(X_train), len(y_train), len(X_test), len(y_test))
print("X_train shape:", X_train.shape, "y_train shape:", y_train.shape, "X_test shape:", X_test.shape, "y_test shape:", y_test.shape)

64 64 17 17
X_train shape: torch.Size([64, 1, 40000, 256]) y_train shape: torch.Size([64, 1, 40000, 256]) X_test shape: torch.Size([17, 1, 40000, 256]) y_test shape: torch.Size([17, 1, 40000, 256])


In [ ]:
data_folder = '/home/mingdayang/mmdetection3d/mapping_test/mini_nuscenes_LC_data'
torch.save({'X_train': X_train, 'y_train': y_train}, os.path.join(data_folder, 'train_data.pt'))
torch.save({'X_test': X_test, 'y_test': y_test}, os.path.join(data_folder, 'test_data.pt'))
print("Saved train/test data to disk.")
